# Credit Risk Prediction — Preprocessing Pipeline

This notebook transforms the raw dataset into a clean, encoded, and scaled feature matrix ready for model training.

**Pipeline steps:**
1. Load raw data
2. Remove outliers and cap extreme values
3. Impute missing values
4. Engineer new features
5. Encode categorical features
6. Train / test split (stratified, 80 / 20)
7. Scale numerical features (fit on train only)
8. Save processed datasets to `data/processed/`

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 1. Data Loading

In [2]:
df = pd.read_csv('../data/raw/credit_risk_dataset.csv')
print(f"At starting: {df.shape[0]} rows, {df.shape[1]} columns")

At starting: 32581 rows, 12 columns


## 2. Outlier Removal & Income Capping

Three types of impossible values are removed before any imputation. `person_income` is winsorized at the 99.5th percentile to reduce the influence of extreme high-income outliers without discarding the rows entirely.

In [3]:
# 2.1 Remove biologically impossible ages (> 100)
old = len(df)
df = df[df['person_age'] <= 100].copy()
print(f'Removed {old - len(df)} rows where person_age > 100')

# 2.2 Remove impossible employment lengths (> 60 years)
old = len(df)
df = df[df['person_emp_length'].isna() | (df['person_emp_length'] <= 60)].copy()
print(f'Removed {old - len(df)} rows where person_emp_length > 60')

# 2.3 Remove rows where employment length exceeds age (logical impossibility)
old = len(df)
mask = df['person_emp_length'].notna() & (df['person_emp_length'] > df['person_age'])
df = df[~mask].copy()
print(f'Removed {old - len(df)} rows where emp_length > age')

# 2.4 Winsorize person_income at the 99.5th percentile to reduce extreme outlier influence
income_cap = df['person_income'].quantile(0.995)
print(f'\nIncome cap (99.5th percentile): {income_cap:,.0f}')
df['person_income'] = df['person_income'].clip(upper=income_cap)

Removed 5 rows where person_age > 100
Removed 2 rows where person_emp_length > 60
Removed 0 rows where emp_length > age

Income cap (99.5th percentile): 300,000


## 3. Missing Value Imputation

Rather than dropping rows (as done during EDA exploration), missing values are imputed here to preserve more training data:
- `person_emp_length` → global median
- `loan_int_rate` → per-`loan_grade` median (preserves the grade–rate relationship)

In [4]:
print('=' * 60)
print('MISSING VALUE IMPUTATION')
print('=' * 60)

# person_emp_length → fill with median
emp_median = df['person_emp_length'].median()
df['person_emp_length'] = df['person_emp_length'].fillna(emp_median)
print(f'person_emp_length: {df["person_emp_length"].isna().sum()} NaN remaining  (filled with median = {emp_median})')

# loan_int_rate → fill with per-grade median to preserve grade-rate relationship
df['loan_int_rate'] = df.groupby('loan_grade')['loan_int_rate'].transform(
    lambda x: x.fillna(x.median())
)
print(f'loan_int_rate: {df["loan_int_rate"].isna().sum()} NaN remaining  (filled with per-grade median)')

print(f'\nTotal missing values remaining: {df.isnull().sum().sum()}')

MISSING VALUE IMPUTATION
person_emp_length: 0 NaN remaining  (filled with median = 4.0)
loan_int_rate: 0 NaN remaining  (filled with per-grade median)

Total missing values remaining: 0


## 4. Feature Engineering

New features are derived to capture relationships not visible in the raw columns.

In [5]:
print('=' * 60)
print('FEATURE ENGINEERING')
print('=' * 60)

# Ratio: income to loan amount — higher ratio = lower default risk
df['income_to_loan_ratio'] = df['person_income'] / df['loan_amnt']

# Ratio: employment length to age — proxy for career stability
df['emp_length_to_age_ratio'] = df['person_emp_length'] / df['person_age']

# Ratio: credit history length to age — proxy for financial maturity
df['cred_hist_to_age_ratio'] = df['cb_person_cred_hist_length'] / df['person_age']

# Life-stage bins
df['age_group'] = pd.cut(
    df['person_age'],
    bins=[0, 25, 35, 45, 60, 100],
    labels=['18-25', '26-35', '36-45', '46-60', '60+']
)

# Income quartile bracket
df['income_bracket'] = pd.qcut(
    df['person_income'], q=4,
    labels=['Low', 'Mid-Low', 'Mid-High', 'High']
)

# Ordinal encoding of loan_grade (A=1 best, G=7 worst)
grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
df['loan_grade_numeric'] = df['loan_grade'].map(grade_map)

# Binary flag: interest rate above median
df['high_interest_flag'] = (df['loan_int_rate'] > df['loan_int_rate'].median()).astype(int)

# Binary flag: prior default on credit bureau file
df['default_history_flag'] = (df['cb_person_default_on_file'] == 'Y').astype(int)

new_features = [
    'income_to_loan_ratio', 'emp_length_to_age_ratio', 'cred_hist_to_age_ratio',
    'age_group', 'income_bracket', 'loan_grade_numeric',
    'high_interest_flag', 'default_history_flag'
]
print('Engineered features added:')
for f in new_features:
    print(f'  + {f}')

FEATURE ENGINEERING
Engineered features added:
  + income_to_loan_ratio
  + emp_length_to_age_ratio
  + cred_hist_to_age_ratio
  + age_group
  + income_bracket
  + loan_grade_numeric
  + high_interest_flag
  + default_history_flag


## 5. Dataset Summary

In [6]:
print('=' * 60)
print('CLEANED DATASET SUMMARY')
print('=' * 60)
print(f'Shape: {df.shape}')

print('\nTarget distribution:')
vc = df['loan_status'].value_counts(normalize=True).mul(100).round(2)
print(f'  No default (0): {vc[0]}%')
print(f'  Default    (1): {vc[1]}%')

print('\nCorrelation of engineered features with loan_status:')
num_new = [
    'income_to_loan_ratio', 'emp_length_to_age_ratio', 'cred_hist_to_age_ratio',
    'loan_grade_numeric', 'high_interest_flag', 'default_history_flag'
]
corr = df[num_new + ['loan_status']].corr()['loan_status'].drop('loan_status')
print(corr.sort_values(key=abs, ascending=False).round(3))

# Save intermediate cleaned dataset (before encoding / scaling)
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/credit_risk_cleaned.csv', index=False)
print(f'\nIntermediate cleaned dataset saved to data/processed/credit_risk_cleaned.csv')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

CLEANED DATASET SUMMARY
Shape: (32574, 20)

Target distribution:
  No default (0): 78.18%
  Default    (1): 21.82%

Correlation of engineered features with loan_status:
loan_grade_numeric         0.373
high_interest_flag         0.242
default_history_flag       0.179
income_to_loan_ratio      -0.176
emp_length_to_age_ratio   -0.086
cred_hist_to_age_ratio    -0.016
Name: loan_status, dtype: float64

Intermediate cleaned dataset saved to data/processed/credit_risk_cleaned.csv
Shape: 32,574 rows x 20 columns


## 6. Encoding Categorical Features

- `loan_grade` and `cb_person_default_on_file` are dropped (replaced by numeric equivalents)
- `person_home_ownership`, `loan_intent`, `age_group`, and `income_bracket` are one-hot encoded

In [7]:
# Drop columns already replaced by engineered numeric equivalents
drop_cols = ['loan_grade', 'cb_person_default_on_file']
df_enc = df.drop(columns=drop_cols)

# One-hot encode nominal and binned categoricals
ohe_cols = ['person_home_ownership', 'loan_intent', 'age_group', 'income_bracket']
df_enc = pd.get_dummies(df_enc, columns=ohe_cols, drop_first=False, dtype=int)

print(f'Shape after encoding: {df_enc.shape}')
print(f'\nAll features ({len(df_enc.columns) - 1} predictors + 1 target):')
for col in df_enc.columns:
    print(f'  {col}')

Shape after encoding: (32574, 33)

All features (32 predictors + 1 target):
  person_age
  person_income
  person_emp_length
  loan_amnt
  loan_int_rate
  loan_status
  loan_percent_income
  cb_person_cred_hist_length
  income_to_loan_ratio
  emp_length_to_age_ratio
  cred_hist_to_age_ratio
  loan_grade_numeric
  high_interest_flag
  default_history_flag
  person_home_ownership_MORTGAGE
  person_home_ownership_OTHER
  person_home_ownership_OWN
  person_home_ownership_RENT
  loan_intent_DEBTCONSOLIDATION
  loan_intent_EDUCATION
  loan_intent_HOMEIMPROVEMENT
  loan_intent_MEDICAL
  loan_intent_PERSONAL
  loan_intent_VENTURE
  age_group_18-25
  age_group_26-35
  age_group_36-45
  age_group_46-60
  age_group_60+
  income_bracket_Low
  income_bracket_Mid-Low
  income_bracket_Mid-High
  income_bracket_High


## 7. Train / Test Split

An 80/20 stratified split is used to maintain the same class ratio in both sets. `random_state=42` ensures reproducibility.

In [8]:
X = df_enc.drop(columns=['loan_status'])
y = df_enc['loan_status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]:,} rows  |  default rate: {y_train.mean():.1%}')
print(f'Test:  {X_test.shape[0]:,} rows  |  default rate: {y_test.mean():.1%}')

Train: 26,059 rows  |  default rate: 21.8%
Test:  6,515 rows  |  default rate: 21.8%


## 8. Feature Scaling

`StandardScaler` is **fit on the training set only** and then applied to both train and test sets. This prevents data leakage from test statistics into the model.

In [9]:
numeric_to_scale = [
    'person_age', 'person_income', 'person_emp_length',
    'loan_amnt', 'loan_int_rate', 'loan_percent_income',
    'cb_person_cred_hist_length', 'income_to_loan_ratio',
    'emp_length_to_age_ratio', 'cred_hist_to_age_ratio'
]

scaler = StandardScaler()
X_train = X_train.copy()
X_test  = X_test.copy()
X_train[numeric_to_scale] = scaler.fit_transform(X_train[numeric_to_scale])
X_test[numeric_to_scale]  = scaler.transform(X_test[numeric_to_scale])

print('StandardScaler fitted on training data and applied to both splits.')
print(f'Scaled {len(numeric_to_scale)} features: {numeric_to_scale}')

StandardScaler fitted on training data and applied to both splits.
Scaled 10 features: ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'income_to_loan_ratio', 'emp_length_to_age_ratio', 'cred_hist_to_age_ratio']


## 9. Save Processed Datasets

Outputs written to `data/processed/`:
- `X_train.csv` / `X_test.csv` — feature matrices
- `y_train.csv` / `y_test.csv` — target labels
- `scaler.pkl` — fitted `StandardScaler` (needed for inference)

In [10]:
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False, header=True)
y_test.to_csv('../data/processed/y_test.csv', index=False, header=True)
joblib.dump(scaler, '../data/processed/scaler.pkl')

print('Saved to data/processed/')
print(f'  X_train.csv  {X_train.shape}')
print(f'  X_test.csv   {X_test.shape}')
print(f'  y_train.csv  {y_train.shape}')
print(f'  y_test.csv   {y_test.shape}')
print(f'  scaler.pkl   StandardScaler')

Saved to data/processed/
  X_train.csv  (26059, 32)
  X_test.csv   (6515, 32)
  y_train.csv  (26059,)
  y_test.csv   (6515,)
  scaler.pkl   StandardScaler
